# 02 — Silver: tipado, limpo, deduplicado

Onde o julgamento é aplicado. Quatro coisas acontecem:

1. Strings viram tipos de verdade; a string literal `'null'` vira um NULL de verdade
2. Datas sentinela são resolvidas
3. `clients` é modelado como dimensão SCD Type 2
4. **`is_bulk_load` é derivado** — a única transformação que muda todas as
   conclusões a jusante

Colunas de data redundantes em string (`created_at_str`, `closed_at_str`)
são descartadas aqui, não na ingestão, para que o Bronze permaneça uma cópia
fiel da fonte.

In [0]:
from pyspark.sql import functions as F, Window

spark.sql("CREATE SCHEMA IF NOT EXISTS silver.bid")

bronze_bids = spark.table("bronze.bid.bids")
bronze_clients = spark.table("bronze.bid.clients")

## Tratamento de nulos

A exportação grava a string de quatro caracteres `'null'`, e `'-'` para uma
data de fechamento ausente. Nenhuma das duas é um NULL para o Spark, então
as duas sobreviveriam silenciosamente a qualquer filtro a jusante.

In [0]:
def denull(df, placeholders=("null", "-", "")):
    """Substitui strings placeholder por NULLs de verdade em todas as colunas string."""
    for c, t in df.dtypes:
        if t == "string":
            df = df.withColumn(
                c, F.when(F.trim(F.col(c)).isin(list(placeholders)), None)
                    .otherwise(F.col(c))
            )
    return df


bids = denull(bronze_bids)
clients = denull(bronze_clients)

## Tipagem

`outcome` é deliberadamente deixado nullable: NULL significa que a proposta
ainda está aberta, um estado distinto de perdida, e não pode colapsar em
`0`. Aproximadamente 28% da tabela está nesse estado.

`to_timestamp` recebe um format explícito em vez de deixar pra inferir. O
parser padrão depende da versão e retorna NULL num erro de parsing em vez
de lançar exceção — exatamente o tipo de perda silenciosa de dado que este
pipeline foi construído para evitar. O assert logo depois falha
ruidosamente se isso algum dia acontecer.

In [0]:
TS_FORMAT = "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"

bids_typed = (
    bids
    .withColumn("bid_id", F.col("bid_id").cast("long"))
    .withColumn("client_id", F.col("client_id").cast("long"))
    .withColumn("created_at", F.to_timestamp("created_at", TS_FORMAT))
    .withColumn("bid_date", F.to_timestamp("bid_date", TS_FORMAT))
    .withColumn("closed_at", F.to_timestamp("closed_at", TS_FORMAT))
    .withColumn("outcome", F.col("outcome").cast("int"))          # NULL = aberta
    .withColumn("is_confirmed_date", F.col("is_confirmed_date").cast("int"))
    .withColumn("contract_value_brl", F.col("contract_value_brl").cast("double"))
    .drop("created_at_str", "closed_at_str")
)

# Guarda: um created_at que era não-NULL antes do cast mas virou NULL depois
# significa que a format string parou de casar com a fonte — falhar
# ruidosamente, não silenciosamente.
bad_created = (
    bids.filter(F.col("created_at").isNotNull())
    .join(bids_typed.filter(F.col("created_at").isNull()), "bid_id", "inner")
    .count()
)
assert bad_created == 0, (
    f"{bad_created} valores de created_at falharam ao parsear com o formato {TS_FORMAT} "
    "— verifique se o formato de data da fonte mudou."
)

## Derivando `is_bulk_load`

Uma grande fração das linhas foi importada em massa durante uma migração de
sistema, em vez de registrada conforme as propostas aconteciam. Elas
compartilham um `created_at` idêntico até o segundo, e se comportam de
forma completamente diferente das propostas orgânicas — uma taxa de vitória
muito mais baixa, concentrada em carteiras específicas.

Sem essa marcação, elas contaminam toda métrica segmentada: o executivo
dono da carteira migrada parece o pior desempenho da empresa, puramente
por causa de como os registros dele foram carregados.

O limiar de 10 é uma decisão de julgamento. Duas propostas registradas no
mesmo segundo é plausível; dez não é.

In [0]:
BULK_THRESHOLD = 10

batch = Window.partitionBy("created_at")

bids_flagged = (
    bids_typed
    .withColumn("_batch_size", F.count("*").over(batch))
    .withColumn("is_bulk_load", (F.col("_batch_size") >= BULK_THRESHOLD).cast("boolean"))
    .drop("_batch_size")
)

## Cobertura de motivo de perda

`competitor_name` carrega um valor padrão gravado sempre que ninguém
completou a apuração pós-proposta. Marcá-lo explicitamente evita que ele
seja contado como um concorrente real em qualquer agregado a jusante — o
que, do contrário, produziria a manchete falsa de que um concorrente leva
a esmagadora maioria das perdas.

In [0]:
PLACEHOLDER_COMPETITOR = "Competitor 1"

bids_clean = (
    bids_flagged
    .withColumn(
        "competitor_is_placeholder",
        (F.col("competitor_name") == F.lit(PLACEHOLDER_COMPETITOR)).cast("boolean"),
    )
    .withColumn("has_loss_reason", F.col("loss_reason").isNotNull())
    .withColumn(
        "bid_status",
        F.when(F.col("outcome") == 1, "won")
         .when(F.col("outcome") == 0, "lost")
         .otherwise("open"),
    )
)

bids_clean.write.format("delta").mode("overwrite").saveAsTable("silver.bid.bids_clean")

## Clients: datas sentinela e SCD Type 2

`2999-12-31` marca um contrato sem prazo definido; mantê-lo como data
colocaria um contrato de 977 anos em qualquer cálculo de duração. Ele vira
NULL, junto com um flag explícito `is_open_ended`.

Renovações são uma segunda linha para o mesmo `client_id`, com um
`start_date` genuinamente posterior ao `end_date` do primeiro contrato —
uma segunda janela de vigência real, não uma duplicata do mesmo dia.
Colapsar para "a linha mais recente" (a abordagem antiga) descartava o
período anterior por completo. Esta versão mantém cada versão e dá a cada
uma um `valid_from` / `valid_to` explícito:

- `valid_from` = o `start_date` daquela versão
- `valid_to` = o `start_date` da *próxima* versão daquele cliente, ou NULL
  se não houver uma — NULL significa **vigente atualmente**, não "desconhecido"
- `is_current` = `valid_to IS NULL`
- `client_sk` = uma chave substituta identificando um par (cliente, versão) —
  `client_id` continua sendo a chave de negócio para joins que só se
  importam com *quem*, não *qual versão*

O Gold decide se quer a versão atual ou a versão que estava vigente quando
uma proposta específica foi feita. O trabalho do Silver termina em tornar
as duas possíveis.

In [0]:
SENTINEL = "2999-12-31"

clients_typed = (
    clients
    .withColumn("client_id", F.col("client_id").cast("long"))
    .withColumn("is_open_ended", (F.col("end_date").startswith(SENTINEL)).cast("boolean"))
    .withColumn(
        "end_date",
        F.when(F.col("end_date").startswith(SENTINEL), None)
         .otherwise(F.to_date("end_date")),
    )
    .withColumn("start_date", F.to_date("start_date"))
)

# Um start_date nulo (o placeholder '-', ~4% das linhas) não pode ser
# posicionado numa sequência de versões de forma confiável; ordenado
# primeiro por convenção em vez de descartado silenciosamente, e end_date
# é o critério de desempate secundário para que duas versões nunca empatem
# — o gap de desempate determinístico que a deduplicação antiga tinha.
version_order = Window.partitionBy("client_id").orderBy(
    F.col("start_date").asc_nulls_first(), F.col("end_date").asc_nulls_last()
)

clients_clean = (
    clients_typed
    .withColumn("valid_from", F.col("start_date"))
    .withColumn("valid_to", F.lead("start_date").over(version_order))  # exclusivo; NULL = vigente
    .withColumn("is_current", F.col("valid_to").isNull())
    .withColumn("client_sk", F.monotonically_increasing_id())
)

# Fazendo DROP antes de escrever, não só overwriteSchema=true: as colunas
# desta tabela mudaram (foram adicionadas valid_from/valid_to/is_current/
# client_sk), e uma definição de tabela do Unity Catalog desatualizada
# pode discordar do novo metadata do Delta mesmo com overwriteSchema
# ligado, lançando DELTA_METADATA_MISMATCH. Um drop + recriação limpa
# evita essa classe inteira de erro — consistente com o padrão de
# "overwrite completo, não incremental" deste pipeline em todo o resto.
spark.sql("DROP TABLE IF EXISTS silver.bid.clients_clean")
clients_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("silver.bid.clients_clean")

# Lê de volta em vez de reusar o DataFrame clients_clean em memória abaixo:
# se duas versões algum dia empatarem exatamente em (start_date, end_date)
# — o gerador antigo produzia duplicatas do mesmo dia antes de ser
# corrigido pra dar às renovações um gap real —, a window function do
# Spark não tem garantia de desempatar da mesma forma em toda
# reavaliação do mesmo plano lazy. A tabela recém-gravada em disco é a
# única coisa com garantia de bater com o que realmente foi persistido.
clients_clean = spark.table("silver.bid.clients_clean")

# Todo client_id deve ter exatamente uma versão vigente — garantido por
# construção (a última linha em cada partição sempre tem valid_to = NULL),
# checado explicitamente mesmo assim porque uma violação silenciosa aqui
# quebraria silenciosamente todo join point-in-time a jusante.
bad_versioning = (
    clients_clean.groupBy("client_id")
    .agg(F.sum(F.col("is_current").cast("int")).alias("n_current"))
    .filter("n_current != 1")
    .count()
)
assert bad_versioning == 0, f"{bad_versioning} client_ids não têm exatamente uma versão vigente"

## Integridade referencial

Uma proposta apontando para um cliente que não existe seria descartada
silenciosamente por um inner join mais tarde. Checar aqui significa que
isso aparece como um número, não como uma contagem de linhas encolhendo
silenciosamente.

A checagem point-in-time abaixo tem expectativa de encontrar uma contagem
não-trivial, diferente de zero — cerca de 6% das propostas. Duas causas
reais, não um bug neste join: alguns clientes têm `start_date` desconhecido
(o placeholder `'-'`) e não podem ser posicionados numa linha do tempo de
jeito nenhum, e algumas propostas foram criadas *antes* do início de
contrato mais antigo conhecido do cliente — uma inconsistência genuína nos
dados de origem que um join simples por `client_id` vinha disfarçando
silenciosamente, ao anexar qualquer versão que calhasse de ser "a mais
recente", data seja lá qual for. O left join no Gold mantém essas propostas
com atributos de cliente NULL, em vez de descartá-las ou chutar.

In [0]:
# Integridade referencial: o cliente existe, de alguma forma? (Qualquer
# versão — validade temporal é uma preocupação separada, checada a seguir.)
orphans = (
    bids_clean.join(clients_clean.select("client_id").distinct(), "client_id", "left_anti").count()
)
print(f"propostas órfãs: {orphans}")

# Cobertura point-in-time: o created_at de cada proposta realmente cai
# dentro da janela [valid_from, valid_to) de alguma versão do seu cliente?
# Um gap aqui significaria que o join point-in-time do Gold descarta os
# atributos de cliente daquela proposta silenciosamente.
coverage_check = (
    bids_clean.alias("b")
    .join(
        clients_clean.alias("c"),
        (F.col("b.client_id") == F.col("c.client_id"))
        & (F.col("b.created_at") >= F.col("c.valid_from").cast("timestamp"))
        & (F.col("c.valid_to").isNull() | (F.col("b.created_at") < F.col("c.valid_to").cast("timestamp"))),
        "left_anti",
    )
)
uncovered = coverage_check.count()
print(f"propostas sem versão de cliente correspondente no created_at: {uncovered}")